# 1. What is K-Nearest Neighbors?

**What is KNN?**
K-Nearest Neighbors (KNN) is a simple, intuitive, and highly effective supervised machine learning algorithm used for both classification and regression.

**Why is it called "K-Nearest Neighbors"?**
When making a prediction for a new, unseen data point, the algorithm looks at the "$K$" closest data points (neighbors) in the training dataset to make its decision.

**Basic Intuition:**
"Tell me who your friends are, and I will tell you who you are."
If a new point is surrounded by mostly Class A points, KNN predicts Class A. If it's surrounded by Class B, it predicts Class B.

**How KNN makes predictions:**
Unlike algorithms that learn a complex mathematical equation (like Linear Regression) or draw a line (like Logistic Regression), KNN simply memorizes the training data. To predict, it calculates the distance between the new point and all memorized points, finds the closest ones, and takes a vote.

**Real-world applications:**
* **Recommendation Systems:** Suggesting movies watched by users similar to you.
* **Handwriting Recognition:** Recognizing handwritten digits based on pixel similarity.
* **Anomaly Detection:** Flagging fraudulent transactions that look very different from normal neighbors.

> Key Idea: KNN does not "learn" a model in the traditional sense during training. It simply memorizes the data and does all the hard work during prediction.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualizing the basic idea
plt.figure(figsize=(6, 4))
# Class A (Blue squares)
plt.scatter([1, 1.5, 2, 2.5], [1, 2, 1.5, 2], c='blue', marker='s', s=100, label='Class A')
# Class B (Red triangles)
plt.scatter([4, 4.5, 5, 5.5], [4, 5, 4.5, 5], c='red', marker='^', s=100, label='Class B')
# New Unknown Point
plt.scatter([3], [3], c='green', marker='o', s=200, edgecolors='black', label='New Point (?)')

plt.title('Basic Intuition of KNN')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.legend()
plt.grid(True)
plt.show()

# If K=3, the new point will look at its 3 closest neighbors to decide its class!


# 2. KNN for Classification

Let's break down how classification works.

* **Input features:** The numerical characteristics of the data (e.g., Height and Weight) used to plot the points in space.
* **Training examples:** The pre-labeled data the model has memorized.
* **Distance from new point:** We calculate exactly how far the new point is from every single training example.
* **Choosing nearest points:** We rank the distances and pick the $K$ points with the smallest distance.
* **Majority voting:** We count the classes of those $K$ points. The class with the most votes wins!


In [ ]:
import numpy as np

# Simple 2D Dataset
X_train = np.array([[1, 2], [1.5, 1.8], [5, 8], [8, 8], [1, 0.6], [9, 11]])
y_train = np.array([0, 0, 1, 1, 0, 1]) # 0=Blue, 1=Red

new_point = np.array([3.5, 4.5])

plt.figure(figsize=(6, 4))
plt.scatter(X_train[y_train==0][:, 0], X_train[y_train==0][:, 1], c='blue', label='Class 0', s=100)
plt.scatter(X_train[y_train==1][:, 0], X_train[y_train==1][:, 1], c='red', label='Class 1', s=100)
plt.scatter(new_point[0], new_point[1], c='green', marker='*', s=300, edgecolors='black', label='New Point')

plt.title('KNN for Classification')
plt.xlabel('X1')
plt.ylabel('X2')
plt.legend()
plt.grid(True)
plt.show()


# 3. Choosing K

What does **$K$** actually mean? 
$K$ is the number of neighbors the algorithm will look at before taking a vote.

* **$K = 1$:** Looks only at the absolute closest single neighbor.
* **$K = 3$:** Looks at the 3 closest neighbors.
* **$K = 5$:** Looks at the 5 closest neighbors.

> Important: We usually choose an **odd** number for $K$ (e.g., 3, 5, 7) in binary classification to prevent tie votes!

**The effect of K:**
* **Small K (e.g., K=1):** The model is very sensitive to noise. If a single red point accidentally ended up in a sea of blue points, a new point right next to it would be classified as red. This leads to jagged, highly complex decisions (**Overfitting**).
* **Large K (e.g., K=50):** The model ignores local noise and looks at the big picture. It creates very smooth decisions. However, if K is too large, the most common class in the dataset will always win, regardless of distance (**Underfitting**).


In [ ]:
# Visualization concept (Conceptual demonstration)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

titles = ['K = 1 (Overfitting / Sensitive to noise)', 
          'K = 5 (Good Balance)', 
          'K = 50 (Underfitting / Too Smooth)']

for i, ax in enumerate(axes):
    ax.scatter(X_train[y_train==0][:, 0], X_train[y_train==0][:, 1], c='blue', s=50)
    ax.scatter(X_train[y_train==1][:, 0], X_train[y_train==1][:, 1], c='red', s=50)
    # Adding a fake "noise" point for demonstration
    ax.scatter([2], [2], c='red', marker='x', s=100) 
    ax.set_title(titles[i])
    ax.set_xticks([])
    ax.set_yticks([])

plt.show()


# 4. Distance Metrics

How do we measure "closeness"? We need a mathematical distance metric.

### Euclidean Distance
The most common metric. It calculates the straight-line distance ("as the crow flies") between two points.

**Formula:**
$$ d = \sqrt{(x_2 - x_1)^2 + (y_2 - y_1)^2} $$

### Manhattan Distance
Calculates distance if you could only travel along grid lines (like a taxi in Manhattan). Useful for high-dimensional data or grid-like spaces.

**Formula:**
$$ d = |x_2 - x_1| + |y_2 - y_1| $$

### Minkowski Distance
A generalized formula. 
* If $p=1$, it becomes Manhattan distance.
* If $p=2$, it becomes Euclidean distance.


In [ ]:
# Calculating distances manually in Python
p1 = np.array([1, 1])
p2 = np.array([4, 5])

# Euclidean
euclidean = np.sqrt(np.sum((p1 - p2)**2))
print(f"Euclidean Distance: {euclidean:.2f}")

# Manhattan
manhattan = np.sum(np.abs(p1 - p2))
print(f"Manhattan Distance: {manhattan:.2f}")


# 5. KNN Prediction Process

Let's walk through the exact steps the algorithm takes to make a prediction:

1. **Select K:** Decide how many neighbors to check (e.g., $K=3$).
2. **Calculate distances:** Measure the distance from the new point to *every single* point in the training set.
3. **Sort distances:** Rank the training points from closest to farthest.
4. **Select K nearest:** Keep only the top $K$ closest points.
5. **Majority class:** Count which class appears most often among those $K$ points.
6. **Assign:** Predict that class for the new point.


In [ ]:
# Manual Prediction Example (K=3)

# 1. Select K
k = 3

# 2. Calculate distances to new_point [3.5, 4.5]
distances = []
for i in range(len(X_train)):
    dist = np.sqrt(np.sum((X_train[i] - new_point)**2))
    distances.append((dist, y_train[i])) # Store distance and class

# 3. Sort distances
distances.sort(key=lambda x: x[0])
print("Sorted Distances (Dist, Class):")
for d, c in distances:
    print(f"Dist: {d:.2f}, Class: {c}")

# 4. Select K nearest
k_nearest = distances[:k]

# 5. Majority vote
classes = [c for d, c in k_nearest]
predicted_class = max(set(classes), key=classes.count)

# 6. Assign
print(f"\nFor K={k}, the predicted class is: {predicted_class}")


# 6. KNN from Scratch

Let's wrap the above logic into a reusable Python function using NumPy.


In [ ]:
from collections import Counter

def knn_predict_scratch(X_train, y_train, x_new, k=3):
    # Predicts the class of a single new data point using KNN.
    
    # 1. Calculate distances from x_new to all points in X_train
    # We use numpy broadcasting for speed
    distances = np.sqrt(np.sum((X_train - x_new)**2, axis=1))
    
    # 2. Sort distances and get the indices of the closest k points
    # argsort returns the indices that would sort the array
    k_indices = np.argsort(distances)[:k]
    
    # 3. Get the labels of those closest k points
    k_nearest_labels = [y_train[i] for i in k_indices]
    
    # 4. Perform majority voting using Counter
    most_common = Counter(k_nearest_labels).most_common(1)
    
    # 5. Return the winning class
    return most_common[0][0]

# Test it out!
test_point = np.array([3.5, 4.5])
pred = knn_predict_scratch(X_train, y_train, test_point, k=3)
print(f"From Scratch Prediction for {test_point}: Class {pred}")


# 7. KNN with Scikit-learn

We don't need to write KNN from scratch in the real world. We use Scikit-learn's `KNeighborsClassifier`.

* `n_neighbors`: This is the $K$ value.


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score

# 1. Load dataset (Synthetic binary classification)
X, y = make_classification(n_samples=200, n_features=2, n_informative=2, n_redundant=0, random_state=42)

# 3. Separate X and y (Done)

# 4. Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Scale features (Standardization)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 6. Create KNN model
model = KNeighborsClassifier(n_neighbors=5)

# 7. Fit model (Memorizes the scaled training data)
model.fit(X_train_scaled, y_train)

# 8. Predict
y_pred = model.predict(X_test_scaled)

# 9. Evaluate
acc = accuracy_score(y_test, y_pred)
print(f"Scikit-learn KNN Accuracy (K=5): {acc:.2f}")


# 8. Why Feature Scaling is Important in KNN

> Important: KNN is highly sensitive to feature scale. This is one of the most critical things to remember about KNN!

**Why?**
KNN calculates physical distance using the Euclidean formula. 

If you have a dataset with:
* **Age:** Ranges from 18 to 60.
* **Salary:** Ranges from 20,000 to 200,000.

When you calculate the distance between two people, the Salary difference might be 15,000, while the Age difference is 5. 
$15000^2$ will completely dwarf $5^2$. The algorithm will basically **ignore Age entirely** and only classify based on Salary, just because the numbers are bigger.

**The Solution:**
Standardize features so they all have a mean of 0 and a variance of 1 (using `StandardScaler`), or normalize them to a 0-1 range (`MinMaxScaler`).


In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score

# Create wildly unscaled data
data = {'Age': [22, 25, 47, 52, 23, 49],
        'Salary': [25000, 30000, 150000, 160000, 27000, 140000],
        'Bought_Car': [0, 0, 1, 1, 0, 1]}
df_scale = pd.DataFrame(data)

X_unscaled = df_scale[['Age', 'Salary']]
y_scale = df_scale['Bought_Car']

# Test on a new point: Age 48, Salary 28000
new_pt = np.array([[48, 28000]]) 

# KNN Without Scaling
knn_unscaled = KNeighborsClassifier(n_neighbors=3).fit(X_unscaled, y_scale)
pred_unscaled = knn_unscaled.predict(new_pt)
print(f"Prediction WITHOUT scaling (Should be 1 based on Age?): {pred_unscaled[0]}") 
# Fails because Salary difference (28000 vs 150000) dominates!

# KNN With Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_unscaled)
new_pt_scaled = scaler.transform(new_pt)

knn_scaled = KNeighborsClassifier(n_neighbors=3).fit(X_scaled, y_scale)
pred_scaled = knn_scaled.predict(new_pt_scaled)
print(f"Prediction WITH scaling: {pred_scaled[0]}")
# Succeeds because both Age and Salary are treated equally!


# 9. Train-Test Split

**Why KNN needs unseen test data:**
KNN memorizes the training data perfectly. If you evaluate a $K=1$ model on its own training data, it will get 100% accuracy because the distance to itself is 0! You **must** evaluate it on a separate, unseen test set to know if it actually works.

* `train_test_split()`: Randomly divides the data.
* `random_state`: Ensures reproducibility.
* `stratify`: Ensures the train and test sets have the same ratio of classes (e.g., 60% dogs, 40% cats in both sets). Highly recommended for classification.


In [ ]:
# Example of stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y # Maintains class proportions
)


# 10. KNN Evaluation Metrics

We evaluate KNN using standard classification metrics:

* **Accuracy:** Total correct / Total predictions. Good for balanced datasets.
* **Precision:** Minimizes False Positives.
* **Recall:** Minimizes False Negatives.
* **F1 Score:** Balance of Precision and Recall.
* **Confusion Matrix:** Shows exactly where the model is making errors.


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

model = KNeighborsClassifier(n_neighbors=5).fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)

print("--- Classification Report ---")
print(classification_report(y_test, y_pred))

print("\n--- Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred))


# 11. Choosing the Best K

> Common Mistake: Choosing K randomly or just sticking to K=5.

To find the best K, we can run a loop, training a model for $K=1, 2, 3... 20$ and recording the accuracy for each.


In [ ]:
k_values = range(1, 20)
accuracies = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    y_pred = knn.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    accuracies.append(acc)

plt.figure(figsize=(8, 4))
plt.plot(k_values, accuracies, marker='o', linestyle='dashed', color='blue')
plt.title('K Value vs Accuracy (On Test Set)')
plt.xlabel('K Value')
plt.ylabel('Accuracy')
plt.xticks(k_values)
plt.grid(True)
plt.show()


**The Problem with this approach:**
We just used our Test Set to pick the best model parameter (K). This means our model has indirectly "seen" the test set, and our final accuracy might be overly optimistic. 

The better approach is **Cross-Validation**.


# 12. KNN and Cross Validation

**Cross-Validation** splits the *training* data into multiple folds. It trains on some folds and validates on the remaining fold, rotating until every fold has been used. This gives a robust estimate of performance without ever touching the true Test Set.


In [ ]:
from sklearn.model_selection import cross_val_score

cv_scores = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    # Perform 5-fold cross validation on TRAINING data only
    scores = cross_val_score(knn, X_train_scaled, y_train, cv=5, scoring='accuracy')
    cv_scores.append(scores.mean()) # Take average of the 5 folds

plt.figure(figsize=(8, 4))
plt.plot(k_values, cv_scores, marker='s', linestyle='-', color='red')
plt.title('K Value vs Cross-Validation Accuracy')
plt.xlabel('K Value')
plt.ylabel('Mean CV Accuracy')
plt.xticks(k_values)
plt.grid(True)
plt.show()

# Best K is the one with the highest CV score!
best_k = k_values[np.argmax(cv_scores)]
print(f"Best K based on Cross-Validation: {best_k}")


# 13. Decision Boundary

A decision boundary is the visual line that separates where the model will predict Class 0 vs Class 1.

Because KNN makes decisions locally based on neighbors, it creates highly irregular, non-linear boundaries.

* **K = 1:** The boundary is very jagged and wraps tightly around every single point (Overfitting).
* **K = 15:** The boundary is much smoother and ignores local anomalies.


In [ ]:
from matplotlib.colors import ListedColormap

def plot_decision_boundary(X, y, k):
    knn = KNeighborsClassifier(n_neighbors=k).fit(X, y)
    
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.1),
                         np.arange(y_min, y_max, 0.1))
    
    Z = knn.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    plt.contourf(xx, yy, Z, alpha=0.3, cmap=ListedColormap(['#AAAAFF', '#FFAAAA']))
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap=ListedColormap(['#0000FF', '#FF0000']), edgecolors='k')
    plt.title(f'Decision Boundary (K = {k})')

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plot_decision_boundary(X_train_scaled, y_train, k=1)
plt.subplot(1, 2, 2)
plot_decision_boundary(X_train_scaled, y_train, k=15)
plt.show()


# 14. KNN for Regression

KNN is usually used for Classification, but it also works for Regression (predicting a number).

**How?**
* **Classification:** Majority vote of the K nearest neighbors.
* **Regression:** The **average** (mean) of the target values of the K nearest neighbors.

`KNeighborsRegressor` is used for this.


In [ ]:
from sklearn.neighbors import KNeighborsRegressor

X_reg = np.array([[1], [2], [3], [4], [5]])
y_reg = np.array([2.1, 3.9, 6.1, 8.2, 9.8])

# If we want to predict for X=2.5, and K=2
# Nearest neighbors are X=2 (y=3.9) and X=3 (y=6.1)
# Prediction = (3.9 + 6.1) / 2 = 5.0

knn_reg = KNeighborsRegressor(n_neighbors=2).fit(X_reg, y_reg)
print(f"Regression prediction for X=2.5: {knn_reg.predict([[2.5]])[0]}")


# 15. KNN Advantages and Disadvantages

### Advantages
* **Simple:** The math and intuition are incredibly easy to understand.
* **No complex training process:** It's a "lazy learner." It just stores data and does nothing until a prediction is needed.
* **Works for both:** Can do Classification and Regression.
* **Useful for smaller datasets:** Highly effective when you have limited but clean data.

### Disadvantages
* **Slow prediction:** For every single prediction, it must calculate the distance to *every* training point. If you have 1 million rows, prediction is very slow.
* **Sensitive to feature scaling:** Fails completely if data isn't scaled.
* **Sensitive to irrelevant features:** If you add 10 random noise columns, the distance calculation is ruined.
* **Sensitive to noise:** Outliers can heavily skew predictions if K is small.
* **Memory intensive:** You have to store the entire training dataset in memory to make predictions.


# 16. Computational Complexity

Why is KNN considered "expensive"?

* **Training time: $O(1)$**
  * Training is instant because the model doesn't calculate any equations. It just saves the dataset in memory.
* **Prediction time: $O(N \times D)$**
  * Where $N$ is the number of training samples and $D$ is the number of features.
  * To predict ONE point, it has to do the distance math against all $N$ points across all $D$ dimensions. This makes it terrible for real-time predictions on massive datasets.


# 17. KNN and the Curse of Dimensionality

**The Curse of Dimensionality:**
As the number of features (dimensions) increases, the volume of the space increases exponentially. The data points become so spread out that *everything* is far away from *everything else*. 

In very high-dimensional space (e.g., 10,000 features), the concept of "distance" loses its meaning. The distance to the nearest neighbor looks almost the same as the distance to the farthest neighbor!

**Solutions:**
* **Feature selection:** Drop useless columns.
* **Dimensionality reduction:** Use techniques like **PCA (Principal Component Analysis)** to squash 1,000 features down to 10 highly informative features before running KNN.


# 18. Complete End-to-End KNN Project

Let's build a clean, complete KNN pipeline using the famous Iris Dataset.


In [ ]:
# 1. Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 2. Load dataset
iris = load_iris()

# 3. Convert to DataFrame
df_iris = pd.DataFrame(iris.data, columns=iris.feature_names)
df_iris['target'] = iris.target

# 4. Inspect dataset
print(df_iris.head(3))

# 5. Basic EDA (Skipped full plots for brevity, but we know it's relatively clean)

# 6. Separate X and y
X = df_iris.drop('target', axis=1)
y = df_iris['target']

# 7. Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 8. Standardize features (CRITICAL!)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 9 & 10. Try several K values using Cross-Validation
k_range = range(1, 21)
cv_scores = []

for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(knn, X_train_scaled, y_train, cv=5, scoring='accuracy')
    cv_scores.append(scores.mean())

# 11. Select best K
best_k = k_range[np.argmax(cv_scores)]
print(f"\nOptimal K found via CV: {best_k}")

# 12. Train final KNN model
final_knn = KNeighborsClassifier(n_neighbors=best_k)
final_knn.fit(X_train_scaled, y_train)

# 13. Make predictions
y_pred = final_knn.predict(X_test_scaled)

# 14-17. Calculate Metrics
print("\n--- Final Model Evaluation ---")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

# 18. Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

# 19. Plot results
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, cmap='Blues', xticklabels=iris.target_names, yticklabels=iris.target_names)
plt.title(f'Confusion Matrix (K={best_k})')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

# 20. Explain the final result
# The model achieved exceptional accuracy on the test set. 
# Using cross-validation ensured we didn't just get lucky on the test set.
# Feature scaling ensured all petal/sepal measurements contributed fairly to the distance calculation.


# 19. Compare Different K Values

Let's visualize the cross-validation curve we just created to see the Underfitting vs Overfitting zones.


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(k_range, cv_scores, marker='o', color='purple')
plt.title('K Value vs Cross-Validation Accuracy (Iris Dataset)')
plt.xlabel('K Value (Number of Neighbors)')
plt.ylabel('Cross-Validation Accuracy')

# Annotations
plt.axvline(best_k, color='green', linestyle='--', label=f'Best K ({best_k})')
plt.text(2, min(cv_scores), 'Overfitting\n(Too sensitive)', color='red')
plt.text(16, min(cv_scores), 'Underfitting\n(Too smooth)', color='red')

plt.legend()
plt.grid(True)
plt.show()

# Explanation: 
# Low K (1-3) can be noisy and overfit.
# High K starts to underfit as it just predicts the majority class.
# The peak represents the best balance.


# 20. Common KNN Mistakes

> Common Mistake: Avoid these pitfalls when using KNN!

1. **Not scaling data:**
   * *Problem:* Features with large numbers dominate the distance.
   * *Better approach:* Always use `StandardScaler` or `MinMaxScaler`.
2. **Choosing K randomly:**
   * *Problem:* You might pick a K that underfits or overfits.
   * *Better approach:* Use cross-validation to find the optimal K.
3. **Using too many irrelevant features:**
   * *Problem:* Irrelevant columns distort the distance calculation (Curse of Dimensionality).
   * *Better approach:* Perform feature selection or PCA first.
4. **Using K = 1 without validation:**
   * *Problem:* Highly susceptible to noise/outliers.
   * *Better approach:* Test a range of K values.
5. **Testing many K values directly on test data:**
   * *Problem:* Data leakage. You are tuning your model to the test set.
   * *Better approach:* Tune K using a validation set or cross-validation on the training set.
6. **Ignoring class imbalance:**
   * *Problem:* If 90% of data is Class A, a large K will always vote for Class A.
   * *Better approach:* Use distance weighting (closer neighbors get heavier votes) or balance the dataset.
7. **Ignoring computational cost:**
   * *Problem:* Deploying KNN on a massive real-time dataset causes massive lag.
   * *Better approach:* Use faster algorithms (like Random Forest) for massive datasets, or use approximate nearest neighbors algorithms.
8. **Using KNN on very high-dimensional data:**
   * *Problem:* Distance metrics lose meaning.
   * *Better approach:* Dimensionality reduction.
9. **Not checking distance metric:**
   * *Problem:* Euclidean might not be best for all data (e.g., categorical or text data).
   * *Better approach:* Try Manhattan or Cosine similarity if applicable.
10. **Not removing noisy/irrelevant points:**
    * *Problem:* KNN memorizes noise.
    * *Better approach:* Clean data thoroughly and remove extreme outliers before training.


# 21. Interview Questions

1. **What is KNN?**
   * A supervised machine learning algorithm that predicts classes or values by looking at the 'K' closest data points in the training set.
2. **Why is KNN called lazy learning?**
   * Because it does not "learn" a mathematical model during the training phase. It simply stores the data and waits until prediction time to do computations.
3. **Why is KNN called instance-based learning?**
   * Because predictions are made based on specific instances (memorized data points) rather than an explicit generalized model.
4. **What does K represent?**
   * The number of nearest neighbors the algorithm looks at to make a vote or average.
5. **What happens when K = 1?**
   * The model perfectly captures the training data but is highly sensitive to noise, leading to jagged decision boundaries and overfitting.
6. **What happens when K is very large?**
   * The model ignores local patterns and becomes too smooth, leading to underfitting. It will just predict the majority class.
7. **Why is feature scaling important?**
   * Because KNN relies on distance (like Euclidean). Features with larger numerical ranges will disproportionately dominate the distance calculation.
8. **Euclidean vs Manhattan distance?**
   * Euclidean is straight-line distance. Manhattan is the sum of absolute differences along axes (grid-like).
9. **How does KNN classify a new point?**
   * Calculates distance to all points, sorts them, picks top K, and takes a majority vote of their classes.
10. **How does KNN perform regression?**
    * It takes the average (mean) of the target values of the K nearest neighbors.
11. **What is the curse of dimensionality?**
    * As the number of features grows, the distance between any two points becomes similar, making distance-based algorithms like KNN ineffective.
12. **Is KNN parametric or non-parametric?**
    * Non-parametric. It makes no underlying assumptions about the distribution of the data (like assuming a linear relationship).
13. **What are the advantages of KNN?**
    * Simple, easy to understand, no training time, works for classification and regression.
14. **What are the disadvantages?**
    * Very slow prediction time for large datasets, memory heavy, sensitive to scale and irrelevant features.
15. **How do you choose the best K?**
    * Using Cross-Validation to test a range of K values and picking the one with the best validation accuracy.
16. **Why use cross-validation?**
    * To ensure the chosen K works well on unseen data, without accidentally tuning the model to the final test set.
17. **What is a decision boundary?**
    * The conceptual line separating regions where the model predicts different classes.
18. **Why can KNN be slow?**
    * At prediction time, it must calculate the distance between the new point and every single point in the training set.
19. **Can KNN work with categorical variables?**
    * Not directly with Euclidean distance. They must be encoded (e.g., One-Hot Encoding), or a different metric like Hamming distance must be used.
20. **How does noisy data affect KNN?**
    * If K is small, noise heavily disrupts predictions. A larger K helps smooth out noise.
21. **How does class imbalance affect KNN?**
    * A larger K will naturally favor the majority class because it simply appears more often in the dataset.
22. **What is distance weighting?**
    * Modifying the vote so that closer neighbors have a stronger vote than neighbors that are further away.
23. **What is `n_neighbors`?**
    * The parameter in Scikit-learn's KNeighborsClassifier that specifies the value of K.
24. **What is `metric` in KNN?**
    * The parameter to change how distance is calculated (e.g., 'euclidean', 'manhattan').
25. **When should KNN be avoided?**
    * With massive datasets (millions of rows) requiring real-time predictions, or datasets with thousands of features.


# 22. Quick Revision Cheat Sheet

| Concept            | Meaning |
| ------------------ | ------- |
| **KNN**                | Predicts based on the closest neighbors. |
| **K**                  | Number of neighbors to check. |
| **Euclidean Distance** | Straight-line distance between points. |
| **Manhattan Distance** | Grid-based absolute distance. |
| **Feature Scaling**    | **CRITICAL**. Use StandardScaler before KNN. |
| **Majority Voting**    | Classification rule: Class with most votes wins. |
| **Decision Boundary**  | Divides feature space into class regions. |
| **Cross Validation**   | Best way to find optimal K without touching test data. |
| **Small K (1-3)**      | Overfitting, jagged boundary, sensitive to noise. |
| **Large K**            | Underfitting, smooth boundary, ignores local patterns. |
| **KNN Classifier**     | Uses majority vote for categories. |
| **KNN Regressor**      | Uses average for continuous numbers. |

### KNN Workflow
1. Data
2. Scale (`StandardScaler`)
3. Choose K range
4. Cross-Validate to find best K
5. Calculate Distance (under the hood)
6. Find Neighbors (under the hood)
7. Vote/Average (under the hood)
8. Final Prediction
9. Evaluation


# 23. Practice Problems

Try to solve these on your own:

1. Calculate the Euclidean and Manhattan distance manually between Point A(0,0) and Point B(3,4).
2. Implement a KNN prediction function from scratch using pure Python/NumPy for a small 5-point dataset.
3. Train a `KNeighborsClassifier` on the `load_wine` dataset.
4. Compare the accuracy of models using K = 1, 3, 5, and 7 on the wine dataset.
5. Apply `StandardScaler` to the wine dataset.
6. Compare the accuracy of the scaled dataset vs the unscaled dataset. Does scaling help?
7. Write a loop to test K from 1 to 30, and plot K vs Accuracy.
8. Refactor your loop to use `cross_val_score` instead of checking against the test set.
9. Generate a confusion matrix for your best model.
10. Use `KNeighborsRegressor` on the `load_boston` or `load_diabetes` dataset.

---

## Next Notebook

`05_naive_bayes.ipynb`

In the next notebook, we will explore Naive Bayes, a classification algorithm based entirely on probability and Bayes' Theorem!
